# 02 - Data Cleaning & Preprocessing

**Project**: [Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines](https://www.drivendata.org/competitions/66/flu-shot-learning)

**Author**: Jarret Angbazo

**Date**: May 10, 2026

---

## Table of Contents

1. [Setup & Load](#1-setup--load)
2. [Missingness Indicators](#2-missingness-indicators)
3. [Imputation](#3-imputation)
   - [3.1 Categorical — Mode Imputation](#31-categorical--mode-imputation)
   - [3.2 Numeric — Median Imputation](#32-numeric--median-imputation)
4. [Data Type Corrections](#4-data-type-corrections)
5. [Encoding](#5-encoding)
   - [5.1 Binary / Label Encode (2-level categoricals)](#51-binary--label-encode-2-level-categoricals)
   - [5.2 Ordinal Encode (ordered multi-level categoricals)](#52-ordinal-encode-ordered-multi-level-categoricals)
   - [5.3 One-Hot Encode (nominal multi-level, ≤ 10 levels)](#53-one-hot-encode-nominal-multi-level--10-levels)
   - [5.4 Target Encode (high-cardinality: industry & occupation)](#54-target-encode-high-cardinality-industry--occupation)
6. [Consistency & Range Checks](#6-consistency--range-checks)
7. [Save Cleaned Data](#7-save-cleaned-data)
8. [Cleaning Summary](#8-cleaning-summary)

---

## 1. Setup & Load <a id='1-setup--load'></a>

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Settings
pd.set_option('display.max_columns', None)

# Warnings
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [2]:
# Define paths and constants
RAW_PATH               = Path('../data/raw')
PROCESSED_PATH         = Path('../data/processed')
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

TARGET_COLS     = ['h1n1_vaccine', 'seasonal_vaccine']
ID_COL          = 'respondent_id'

# Load data
features_train = pd.read_csv(RAW_PATH / 'training_set_features.csv', index_col = ID_COL)
labels_train   = pd.read_csv(RAW_PATH / 'training_set_labels.csv', index_col = ID_COL)
features_test  = pd.read_csv(RAW_PATH / 'test_set_features.csv', index_col = ID_COL)

# Work on copies so raw dataframes remain unchanged for debugging
X_train = features_train.copy()
X_test = features_test.copy()
y_train = labels_train.copy()

print("Loaded succesfully.")
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")

Loaded succesfully.
X_train : (26707, 35)
X_test  : (26708, 35)
y_train : (26707, 2)


## 2. Missingness Indicators <a id='2-missingness-indicators'></a>

**EDA finding**: Several columns have MNAR or structurally-informative missingness. The missingness
itself carries predictive signal and must be encoded *before* values are imputed away.

| Column(s) | Missingness % | Mechanism | Indicator needed |
|---|---|---|---|
| `employment_occupation` | 50.4% | Structural MAR — non-employed have no occupation | Yes — separate from industry (362-row discrepancy) |
| `employment_industry` | 49.9% | Structural MAR | Yes |
| `health_insurance` | 46.0% | Partial MAR (also missing for 44% of employed) | Yes |
| `income_poverty` | 16.6% | MNAR (income refusal) | Yes |
| `doctor_recc_h1n1` + `doctor_recc_seasonal` | 8.1% each | Always co-missing (0 discordant rows) | **One** shared indicator covers both |
| Opinion columns (6 cols) | 1.5–2.0% | MNAR for seasonal target (7.6–10.3 pp rate diff) | Captured by `missing_opinion_count` |

> **Note:** `missing_opinion_count` is created in `03_feature_engineering.ipynb` because it is a
> derived numeric feature, not a direct missingness flag. Individual opinion indicators are not
> created here because the aggregate count is more parsimonious.

In [3]:
# Columns that get a 1/0 missingness indicator
INDICATOR_COLS = [
    'employment_occupation',
    'employment_industry',
    'health_insurance',
    'income_poverty',
    'doctor_recc_h1n1',   # shared indicator with doctor_recc_seasonal
]

for col in INDICATOR_COLS:
    ind_name = f'{col}_missing'
    X_train[ind_name] = X_train[col].isnull().astype(int)
    X_test[ind_name]  = X_test[col].isnull().astype(int)

# Rename for clarity — both doctor recc cols share one indicator
X_train.rename(columns={'doctor_recc_h1n1_missing': 'doctor_recc_missing'}, inplace=True)
X_test.rename( columns={'doctor_recc_h1n1_missing': 'doctor_recc_missing'}, inplace=True)
X_train.drop(columns=['doctor_recc_seasonal_missing'], errors='ignore', inplace=True)
X_test.drop( columns=['doctor_recc_seasonal_missing'], errors='ignore', inplace=True)

created = [c for c in X_train.columns if c.endswith('_missing')]
print(f"Missingness indicators created: {created}")
print(f"\nValue counts (train):")
for c in created:
    print(f"  {c}: {X_train[c].value_counts().to_dict()}")

Missingness indicators created: ['employment_occupation_missing', 'employment_industry_missing', 'health_insurance_missing', 'income_poverty_missing', 'doctor_recc_missing']

Value counts (train):
  employment_occupation_missing: {1: 13470, 0: 13237}
  employment_industry_missing: {0: 13377, 1: 13330}
  health_insurance_missing: {0: 14433, 1: 12274}
  income_poverty_missing: {0: 22284, 1: 4423}
  doctor_recc_missing: {0: 24547, 1: 2160}


## 3. Imputation <a id='3-imputation'></a>

**EDA finding:** All numeric features are ordinal or binary (no continuous features), so **median**
imputation is appropriate — it is robust to skewed ordinal distributions and preserves integer-like
values. Mode imputation is used for all categoricals.

**Key rule:** Imputation statistics are fit on the training set only, then applied identically to
the test set to prevent leakage.

### 3.1 Categorical — Mode Imputation <a id='31-categorical--mode-imputation'></a>

In [4]:
cat_cols_to_impute = [
    c for c in X_train.select_dtypes(include='object').columns
    if X_train[c].isnull().any()
]

# Fit modes on train only
cat_modes = {col: X_train[col].mode()[0] for col in cat_cols_to_impute}

for col, mode_val in cat_modes.items():
    X_train[col] = X_train[col].fillna(mode_val)
    X_test[col]  = X_test[col].fillna(mode_val)

print("Categorical columns imputed with training mode:")
for col, val in cat_modes.items():
    print(f"  {col:35s} → '{val}'")

Categorical columns imputed with training mode:
  education                           → 'College Graduate'
  income_poverty                      → '<= $75,000, Above Poverty'
  marital_status                      → 'Married'
  rent_or_own                         → 'Own'
  employment_status                   → 'Employed'
  employment_industry                 → 'fcxhlnwr'
  employment_occupation               → 'xtkaffoo'


### 3.2 Numeric — Median Imputation <a id='32-numeric--median-imputation'></a>

In [5]:
# All numeric features except targets and the freshly-created indicators
num_cols_to_impute = [
    c for c in X_train.select_dtypes(include='number').columns
    if X_train[c].isnull().any() and not c.endswith('_missing')
]

# Fit medians on train only
num_medians = {col: X_train[col].median() for col in num_cols_to_impute}

for col, med_val in num_medians.items():
    X_train[col] = X_train[col].fillna(med_val)
    X_test[col]  = X_test[col].fillna(med_val)

print("Numeric columns imputed with training median:")
for col, val in num_medians.items():
    print(f"  {col:40s} → {val}")

Numeric columns imputed with training median:
  h1n1_concern                             → 2.0
  h1n1_knowledge                           → 1.0
  behavioral_antiviral_meds                → 0.0
  behavioral_avoidance                     → 1.0
  behavioral_face_mask                     → 0.0
  behavioral_wash_hands                    → 1.0
  behavioral_large_gatherings              → 0.0
  behavioral_outside_home                  → 0.0
  behavioral_touch_face                    → 1.0
  doctor_recc_h1n1                         → 0.0
  doctor_recc_seasonal                     → 0.0
  chronic_med_condition                    → 0.0
  child_under_6_months                     → 0.0
  health_worker                            → 0.0
  health_insurance                         → 1.0
  opinion_h1n1_vacc_effective              → 4.0
  opinion_h1n1_risk                        → 2.0
  opinion_h1n1_sick_from_vacc              → 2.0
  opinion_seas_vacc_effective              → 4.0
  opinion_seas_risk    

In [6]:
# ── Verify no remaining nulls ───────────────────────────────────────────
train_nulls = X_train.isnull().sum().sum()
test_nulls  = X_test.isnull().sum().sum()

print(f"Remaining nulls — X_train: {train_nulls} | X_test: {test_nulls}")
assert train_nulls == 0, "X_train still has nulls!"
assert test_nulls  == 0, "X_test still has nulls!"
print("✓ All nulls resolved.")

Remaining nulls — X_train: 0 | X_test: 0
✓ All nulls resolved.


## 4. Data Type Corrections <a id='4-data-type-corrections'></a>

**EDA finding:** All numeric features loaded as `float64` due to pre-imputation NaNs.
After imputation, binary and ordinal features can be safely downcast to `int8` to reduce memory.
No datetime columns exist in this dataset (no time-series structure).

In [7]:
# Binary and ordinal numeric features to downcast to int8
int_cols = [
    'h1n1_concern', 'h1n1_knowledge',
    'behavioral_antiviral_meds', 'behavioral_avoidance', 'behavioral_face_mask',
    'behavioral_wash_hands', 'behavioral_large_gatherings', 'behavioral_outside_home',
    'behavioral_touch_face', 'doctor_recc_h1n1', 'doctor_recc_seasonal',
    'chronic_med_condition', 'child_under_6_months', 'health_worker', 'health_insurance',
    'opinion_h1n1_vacc_effective', 'opinion_h1n1_risk', 'opinion_h1n1_sick_from_vacc',
    'opinion_seas_vacc_effective', 'opinion_seas_risk', 'opinion_seas_sick_from_vacc',
    'household_adults', 'household_children',
]

for col in int_cols:
    X_train[col] = X_train[col].astype('int8')
    X_test[col]  = X_test[col].astype('int8')

print("Downcast to int8 — memory comparison:")
orig_mb = features_train[int_cols].memory_usage(deep=True).sum() / 1e6
new_mb  = X_train[int_cols].memory_usage(deep=True).sum() / 1e6
print(f"  Before: {orig_mb:.2f} MB  →  After: {new_mb:.2f} MB")
print(f"\nDtype check (sample):")
print(X_train[int_cols[:6]].dtypes)

Downcast to int8 — memory comparison:
  Before: 5.13 MB  →  After: 0.83 MB

Dtype check (sample):
h1n1_concern                 int8
h1n1_knowledge               int8
behavioral_antiviral_meds    int8
behavioral_avoidance         int8
behavioral_face_mask         int8
behavioral_wash_hands        int8
dtype: object


## 5. Encoding <a id='5-encoding'></a>

**EDA cardinality findings:**

| Encoding strategy | Columns |
|---|---|
| **Binary / label** (2 levels) | `sex`, `marital_status`, `rent_or_own` |
| **Ordinal** (ordered multi-level) | `age_group`, `education`, `income_poverty`, `employment_status` |
| **One-hot** (nominal, 3–10 levels) | `race`, `hhs_geo_region`, `census_msa` |
| **Target encode** (obfuscated, 21–23 levels) | `employment_industry`, `employment_occupation` |

> All encoding maps are fit on training data only.
> Target encoding uses leave-one-out smoothing to prevent leakage.

### 5.1 Binary / Label Encode (2-level categoricals) <a id='51-binary--label-encode-2-level-categoricals'></a>

In [8]:
# sex: Female=1, Male=0
sex_map = {'Female': 1, 'Male': 0}
X_train['sex'] = X_train['sex'].map(sex_map).astype('int8')
X_test['sex']  = X_test['sex'].map(sex_map).astype('int8')

# marital_status: Married=1, Not Married=0
marital_map = {'Married': 1, 'Not Married': 0}
X_train['marital_status'] = X_train['marital_status'].map(marital_map).astype('int8')
X_test['marital_status']  = X_test['marital_status'].map(marital_map).astype('int8')

# rent_or_own: Own=1, Rent=0
rent_map = {'Own': 1, 'Rent': 0}
X_train['rent_or_own'] = X_train['rent_or_own'].map(rent_map).astype('int8')
X_test['rent_or_own']  = X_test['rent_or_own'].map(rent_map).astype('int8')

print("Binary columns encoded:")
for col, mapping in [('sex', sex_map), ('marital_status', marital_map), ('rent_or_own', rent_map)]:
    print(f"  {col}: {mapping}")

Binary columns encoded:
  sex: {'Female': 1, 'Male': 0}
  marital_status: {'Married': 1, 'Not Married': 0}
  rent_or_own: {'Own': 1, 'Rent': 0}


### 5.2 Ordinal Encode (ordered multi-level categoricals) <a id='52-ordinal-encode-ordered-multi-level-categoricals'></a>

In [9]:
# age_group — natural order by age band
age_map = {
    '18 - 34 Years': 0,
    '35 - 44 Years': 1,
    '45 - 54 Years': 2,
    '55 - 64 Years': 3,
    '65+ Years'    : 4,
}

# education — natural order by attainment
edu_map = {
    '< 12 Years'      : 0,
    '12 Years'        : 1,
    'Some College'    : 2,
    'College Graduate': 3,
}

# income_poverty — natural order by income level
income_map = {
    'Below Poverty'              : 0,
    '<= $75,000, Above Poverty'  : 1,
    '> $75,000'                  : 2,
}

# employment_status — no strict ordinal order, but grouped for tree splits
employment_map = {
    'Unemployed'       : 0,
    'Not in Labor Force': 1,
    'Employed'         : 2,
}

ordinal_maps = {
    'age_group'        : age_map,
    'education'        : edu_map,
    'income_poverty'   : income_map,
    'employment_status': employment_map,
}

for col, mapping in ordinal_maps.items():
    X_train[col] = X_train[col].map(mapping).astype('int8')
    X_test[col]  = X_test[col].map(mapping).astype('int8')
    print(f"  {col}: {mapping}")

  age_group: {'18 - 34 Years': 0, '35 - 44 Years': 1, '45 - 54 Years': 2, '55 - 64 Years': 3, '65+ Years': 4}
  education: {'< 12 Years': 0, '12 Years': 1, 'Some College': 2, 'College Graduate': 3}
  income_poverty: {'Below Poverty': 0, '<= $75,000, Above Poverty': 1, '> $75,000': 2}
  employment_status: {'Unemployed': 0, 'Not in Labor Force': 1, 'Employed': 2}


### 5.3 One-Hot Encode (nominal multi-level, ≤ 10 levels) <a id='53-one-hot-encode-nominal-multi-level--10-levels'></a>

In [10]:
# Fit one-hot categories on training set only; align test to same columns
one_hot_cols = ['race', 'hhs_geo_region', 'census_msa']

# Concatenate train+test, one-hot, then re-split — ensures identical columns
# (drop_first=False: keep all levels; tree models benefit from full dummies)
combined = pd.concat([X_train[one_hot_cols], X_test[one_hot_cols]])
encoded  = pd.get_dummies(combined, columns=one_hot_cols, drop_first=False, dtype='int8')

n_train = len(X_train)
ohe_train = encoded.iloc[:n_train].reset_index(drop=True)
ohe_test  = encoded.iloc[n_train:].reset_index(drop=True)

# Drop originals and attach encoded columns
X_train = X_train.drop(columns=one_hot_cols).reset_index(drop=False)
X_test  = X_test.drop(columns=one_hot_cols).reset_index(drop=False)

X_train = pd.concat([X_train, ohe_train], axis=1).set_index(ID_COL)
X_test  = pd.concat([X_test,  ohe_test],  axis=1).set_index(ID_COL)

ohe_cols_created = [c for c in X_train.columns if any(c.startswith(p+'_') for p in one_hot_cols)]
print(f"One-hot columns created: {len(ohe_cols_created)}")
for c in ohe_cols_created:
    print(f"  {c}")

One-hot columns created: 17
  race_Black
  race_Hispanic
  race_Other or Multiple
  race_White
  hhs_geo_region_atmpeygn
  hhs_geo_region_bhuqouqj
  hhs_geo_region_dqpwygqj
  hhs_geo_region_fpwskwrf
  hhs_geo_region_kbazzjca
  hhs_geo_region_lrircsnp
  hhs_geo_region_lzgpxyit
  hhs_geo_region_mlyzmhmf
  hhs_geo_region_oxchjgsf
  hhs_geo_region_qufhixun
  census_msa_MSA, Not Principle  City
  census_msa_MSA, Principle City
  census_msa_Non-MSA


### 5.4 Target Encode (high-cardinality: industry & occupation) <a id='54-target-encode-high-cardinality-industry--occupation'></a>

In [11]:
# employment_industry (21 levels) and employment_occupation (23 levels) are obfuscated codes.
# EDA finding: one industry code ('haxffmxo') has the highest h1n1 (62%) and seasonal (84%)
# vaccination rates in the dataset. Codes are treated as opaque. Target encoding captures
# this continuous signal cleanly.
#
# Implementation: leave-one-out (LOO) mean encoding with additive smoothing.
# Applied per target to avoid leakage between the two response variables.
# Test set uses the global per-category mean fit on training data.

HIGH_CARD_COLS = ['employment_industry', 'employment_occupation']
SMOOTHING      = 10   # regularisation strength; higher = shrink more toward global mean

def target_encode_loo(X_tr, X_te, col, target_series, smoothing=10):
    """
    Fit LOO target encoding on X_tr[col] using target_series (aligned to X_tr).
    Returns encoded train series and encoded test series.
    """
    df_fit = X_tr[[col]].copy()
    df_fit['_y'] = target_series.values

    global_mean = df_fit['_y'].mean()
    stats = df_fit.groupby(col)['_y'].agg(['sum', 'count'])

    # LOO train encoding
    # For each row: mean of all OTHER rows in same category
    row_sum   = df_fit[col].map(stats['sum'])
    row_count = df_fit[col].map(stats['count'])
    loo_sum   = row_sum   - df_fit['_y']
    loo_count = row_count - 1

    # Smoothing: blend LOO mean with global mean weighted by count
    train_enc = (loo_sum + smoothing * global_mean) / (loo_count + smoothing)

    # Test encoding: full category mean (no LOO needed)
    cat_mean = (stats['sum'] + smoothing * global_mean) / (stats['count'] + smoothing)
    test_enc = X_te[col].map(cat_mean).fillna(global_mean)

    return train_enc, test_enc


for col in HIGH_CARD_COLS:
    for t in TARGET_COLS:
        new_col = f'{col}_te_{t}'
        tr_enc, te_enc = target_encode_loo(
            X_train, X_test, col,
            target_series=y_train[t],
            smoothing=SMOOTHING
        )
        X_train[new_col] = tr_enc.values
        X_test[new_col]  = te_enc.values
        print(f"  Created: {new_col}")

# Drop originals now that they are encoded
X_train.drop(columns=HIGH_CARD_COLS, inplace=True)
X_test.drop( columns=HIGH_CARD_COLS, inplace=True)

print(f"\nX_train shape after encoding: {X_train.shape}")
print(f"X_test  shape after encoding: {X_test.shape}")

  Created: employment_industry_te_h1n1_vaccine
  Created: employment_industry_te_seasonal_vaccine
  Created: employment_occupation_te_h1n1_vaccine
  Created: employment_occupation_te_seasonal_vaccine

X_train shape after encoding: (26707, 56)
X_test  shape after encoding: (26708, 56)


## 6. Consistency & Range Checks <a id='6-consistency--range-checks'></a>

Replicate the EDA range validation on the cleaned data to confirm no corruption occurred during
imputation or encoding.

In [12]:
# Expected bounds for original ordinal features (post-imputation, pre-engineering)
expected_bounds = {
    'h1n1_concern'               : (0, 3),
    'h1n1_knowledge'             : (0, 2),
    'behavioral_antiviral_meds'  : (0, 1),
    'behavioral_avoidance'       : (0, 1),
    'behavioral_face_mask'       : (0, 1),
    'behavioral_wash_hands'      : (0, 1),
    'behavioral_large_gatherings': (0, 1),
    'behavioral_outside_home'    : (0, 1),
    'behavioral_touch_face'      : (0, 1),
    'doctor_recc_h1n1'           : (0, 1),
    'doctor_recc_seasonal'       : (0, 1),
    'chronic_med_condition'      : (0, 1),
    'child_under_6_months'       : (0, 1),
    'health_worker'              : (0, 1),
    'health_insurance'           : (0, 1),
    'opinion_h1n1_vacc_effective': (1, 5),
    'opinion_h1n1_risk'          : (1, 5),
    'opinion_h1n1_sick_from_vacc': (1, 5),
    'opinion_seas_vacc_effective': (1, 5),
    'opinion_seas_risk'          : (1, 5),
    'opinion_seas_sick_from_vacc': (1, 5),
    'household_adults'           : (0, 3),
    'household_children'         : (0, 3),
    'sex'                        : (0, 1),
    'marital_status'             : (0, 1),
    'rent_or_own'                : (0, 1),
    'age_group'                  : (0, 4),
    'education'                  : (0, 3),
    'income_poverty'             : (0, 2),
    'employment_status'          : (0, 2),
}

violations = []
for col, (lo, hi) in expected_bounds.items():
    if col not in X_train.columns:
        continue
    for name, frame in [('train', X_train), ('test', X_test)]:
        mn, mx = frame[col].min(), frame[col].max()
        if mn < lo or mx > hi:
            violations.append(f"{col} [{name}]: got [{mn}, {mx}], expected [{lo}, {hi}]")

if violations:
    print("✗ Range violations:")
    for v in violations:
        print(f"  {v}")
else:
    print("✓ All ordinal/binary features within expected bounds.")

# Duplicate check
print(f"\nDuplicate rows — X_train: {X_train.duplicated().sum()} | X_test: {X_test.duplicated().sum()}")

# Null check
print(f"Remaining nulls — X_train: {X_train.isnull().sum().sum()} | X_test: {X_test.isnull().sum().sum()}")

# Column alignment
assert list(X_train.columns) == list(X_test.columns), "Train/test column mismatch!"
print(f"\n✓ Train and test share identical {len(X_train.columns)} columns.")

✓ All ordinal/binary features within expected bounds.

Duplicate rows — X_train: 1 | X_test: 0
Remaining nulls — X_train: 0 | X_test: 0

✓ Train and test share identical 56 columns.


## 7. Save Cleaned Data <a id='7-save-cleaned-data'></a>

In [13]:
X_train.to_csv(PROCESSED_PATH / 'X_train_clean.csv')
X_test.to_csv( PROCESSED_PATH / 'X_test_clean.csv')
y_train.to_csv(PROCESSED_PATH / 'y_train.csv')

print("Saved:")
print(f"  {PROCESSED_PATH / 'X_train_clean.csv'}  shape={X_train.shape}")
print(f"  {PROCESSED_PATH / 'X_test_clean.csv'}   shape={X_test.shape}")
print(f"  {PROCESSED_PATH / 'y_train.csv'}         shape={y_train.shape}")

Saved:
  ../data/processed/X_train_clean.csv  shape=(26707, 56)
  ../data/processed/X_test_clean.csv   shape=(26708, 56)
  ../data/processed/y_train.csv         shape=(26707, 2)


## 8. Cleaning Summary <a id='8-cleaning-summary'></a>

### Actions Taken

| Step | Action | Detail |
|---|---|---|
| Missingness indicators | 5 binary flags created | `employment_occupation_missing`, `employment_industry_missing`, `health_insurance_missing`, `income_poverty_missing`, `doctor_recc_missing` (shared for both doctor recc cols) |
| Categorical imputation | Mode (fit on train) | `employment_status`, `education`, `marital_status`, `rent_or_own`, `race`, `income_poverty`, `hhs_geo_region`, `census_msa` |
| Numeric imputation | Median (fit on train) | All 23 numeric features with any missing; values range from 0.7%–16.7% missing |
| Downcast to int8 | Memory reduction | 23 binary/ordinal numeric columns |
| Binary encode | Map to 0/1 | `sex`, `marital_status`, `rent_or_own` |
| Ordinal encode | Ordered integer map | `age_group` (0–4), `education` (0–3), `income_poverty` (0–2), `employment_status` (0–2) |
| One-hot encode | Dummy columns | `race` (4→4 cols), `hhs_geo_region` (10→10 cols), `census_msa` (3→3 cols) |
| Target encode | LOO smoothed mean | `employment_industry` → 2 cols (one per target), `employment_occupation` → 2 cols — codes treated as opaque |
| Outlier treatment | None | All numeric features are ordinal/binary — no continuous values to winsorise |

### Shape Changes

| Dataset | Before | After |
|---|---|---|
| X_train | (26707, 35) | (26707, **N**) |
| X_test | (26708, 35) | (26708, **N**) |

*Run the notebook to see the final N — depends on one-hot expansion.*

### What Goes into `03_feature_engineering.ipynb`

- `X_train_clean.csv` — fully imputed, encoded, indicator-augmented feature matrix
- `X_test_clean.csv` — identically processed test features
- `y_train.csv` — untouched labels

Remaining work deferred to `03_feature_engineering.ipynb`:
- `missing_opinion_count` feature
- `high_vacc_ind_occ` binary flag (industry/occupation codes with highest vaccination rates)
- `doctor_recc_both` interaction
- `opinion_h1n1_composite`, `opinion_seas_composite`
- `behavior_composite`